In [1]:
import os
from google.colab import userdata

# ==========================================
# ⚙️ CONFIGURAÇÕES DA AULA & SEGURANÇA
# ==========================================
AULA_ATUAL = "AULA_03"
NOME_REPO = "Residencia-Trilhas-em-Tecnologias-IA-Generativa-RAG"
USUARIO_GITHUB = "EridalgoRamos"
EMAIL_GITHUB = "EridalgoRamos@users.noreply.github.com"

print("🔐 Acessando o cofre de chaves do Colab...")
try:
    # Resgatando as chaves secretas do Colab
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

    # Passando variáveis para o ambiente do sistema
    os.environ['AULA_ATUAL'] = AULA_ATUAL
    os.environ['NOME_REPO'] = NOME_REPO
    os.environ['USUARIO_GITHUB'] = USUARIO_GITHUB
    os.environ['EMAIL_GITHUB'] = EMAIL_GITHUB
    os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY

    print(f"✅ Chaves carregadas com sucesso! Preparando ambiente para a {AULA_ATUAL}...")
except userdata.SecretNotFoundError as e:
    print(f"❌ Erro: O segredo não foi encontrado. Certifique-se de que o nome no painel esquerdo está idêntico a este: {e}")

🔐 Acessando o cofre de chaves do Colab...
✅ Chaves carregadas com sucesso! Preparando ambiente para a AULA_03...


In [2]:
%%bash
# Verifica se a pasta do repositório já existe no Colab
if [ ! -d "$NOME_REPO" ]; then
  echo "📥 Baixando o repositório pela primeira vez..."
  # Clone blindado usando o token invisível
  git clone https://$GITHUB_TOKEN@github.com/$USUARIO_GITHUB/$NOME_REPO.git
else
  echo "🔄 Repositório já existe. Sincronizando últimas atualizações..."
  cd $NOME_REPO
  git pull origin main
fi

# Cria a pasta da aula atual (AULA_03) automaticamente
mkdir -p "$NOME_REPO/$AULA_ATUAL"
echo "📂 Pasta $AULA_ATUAL pronta para uso!"

📥 Baixando o repositório pela primeira vez...
📂 Pasta AULA_03 pronta para uso!


Cloning into 'Residencia-Trilhas-em-Tecnologias-IA-Generativa-RAG'...


In [3]:
%%bash
cd $NOME_REPO

# Só cria o ambiente virtual se a pasta 'venv' não existir
if [ ! -d "venv" ]; then
  echo "🛠️ Criando Ambiente Virtual (Isso pode levar uns segundos)..."
  pip install virtualenv -q
  python3 -m virtualenv venv
else
  echo "✅ Ambiente Virtual já existe."
fi

# Ativa o ambiente e instala as dependências
source venv/bin/activate
echo "📦 Instalando/Atualizando dependências do requirements.txt..."
pip install -r requirements.txt -q

echo "🚀 Ambiente configurado! Tudo pronto para começarmos a programar a $AULA_ATUAL!"

🛠️ Criando Ambiente Virtual (Isso pode levar uns segundos)...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 30.0 MB/s eta 0:00:00
created virtual environment CPython3.12.13.final.0-64-x86_64 in 548ms
  creator CPython3Posix(dest=/content/Residencia-Trilhas-em-Tecnologias-IA-Generativa-RAG/venv, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.2.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator
📦 Instalando/Atualizando dependências do requirements.txt...
🚀 Ambiente configurado! Tudo pronto para começarmos a programar a AULA_03!


In [4]:
import requests
import os

# 1. Puxa a chave secretamente da memória do sistema (sem expor no código!)
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY')

texto_para_testar = "O algoritmo do Twitter impacta o espaço público."

print(f"Gerando embeddings para o texto: '{texto_para_testar}'...\n")

# 2. Faz a requisição para a API
response = requests.post(
  "https://openrouter.ai/api/v1/embeddings",
  headers={
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
  },
  json={
    "model": "openai/text-embedding-3-small",
    "input": texto_para_testar
  }
)

dados = response.json()

# 3. Extrai e imprime o resultado
vetor_embedding = dados["data"][0]["embedding"]

print("✅ Comportamento do Embedding verificado com sucesso!")
print(f"Tamanho do vetor (dimensões): {len(vetor_embedding)}")
print(f"Primeiros 5 números do vetor gerado: {vetor_embedding[:5]}")

Gerando embeddings para o texto: 'O algoritmo do Twitter impacta o espaço público.'...

✅ Comportamento do Embedding verificado com sucesso!
Tamanho do vetor (dimensões): 1536
Primeiros 5 números do vetor gerado: [0.022430419921875, -0.025238037109375, -0.005847930908203125, 0.056396484375, 0.029815673828125]


In [5]:
%%bash
cd $NOME_REPO
source venv/bin/activate

echo "📦 Instalando bibliotecas de visualização e dados (Pandas, Scikit-Learn, Plotly)..."
pip install pandas scikit-learn plotly -q
echo "✅ Pronto para rodar o gráfico!"

📦 Instalando bibliotecas de visualização e dados (Pandas, Scikit-Learn, Plotly)...
✅ Pronto para rodar o gráfico!


In [6]:
import os
import math
import requests
import pandas as pd
from sklearn.decomposition import PCA
import plotly.express as px

# ==========================================
# 1. Funções Matemáticas (Euclidiana e Cosseno)
# ==========================================
def distancia_euclidiana(embedding_a, embedding_b):
    if len(embedding_a) != len(embedding_b):
        raise ValueError("Os embeddings devem ter a mesma dimensão.")
    soma_diferencas_quadrado = sum((a - b) ** 2 for a, b in zip(embedding_a, embedding_b))
    return math.sqrt(soma_diferencas_quadrado)

def distancia_cosseno(embedding_a, embedding_b):
    if len(embedding_a) != len(embedding_b):
        raise ValueError("Os embeddings devem ter a mesma dimensão.")
    produto_escalar = sum(a * b for a, b in zip(embedding_a, embedding_b))
    norma_a = math.sqrt(sum(a ** 2 for a in embedding_a))
    norma_b = math.sqrt(sum(b ** 2 for b in embedding_b))

    if norma_a == 0 or norma_b == 0:
        return 1.0
    return 1.0 - (produto_escalar / (norma_a * norma_b))

# ==========================================
# 2. Configuração de Segurança e API
# ==========================================
print("--- TESTE COM TERMOS REAIS E GRÁFICO 3D ---")

# Puxa a chave secretamente do cofre do Colab que configuramos antes
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY')
termos = ["gato", "felino", "cachorro", "carro", "caminhão", "moto", "banana", "maça", "goiaba"]
dicionario_embeddings = {}

def gerar_embedding_termo(texto):
    response = requests.post(
        "https://openrouter.ai/api/v1/embeddings",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": "openai/text-embedding-3-small",
            "input": texto
        }
    )
    return response.json()["data"][0]["embedding"]

# ==========================================
# 3. Gerando os Vetores e Comparando
# ==========================================
print("⏳ Gerando vetores na API (isso pode levar alguns segundos)...\n")
for termo in termos:
    dicionario_embeddings[termo] = gerar_embedding_termo(termo)

pares = [
    ("gato", "felino"),
    ("gato", "cachorro"),
    ("gato", "carro"),
    ("carro", "moto"),
    ("banana", "maça"),
    ("caminhão", "goiaba")
]

for p1, p2 in pares:
    emb_1, emb_2 = dicionario_embeddings[p1], dicionario_embeddings[p2]
    dist_cos = distancia_cosseno(emb_1, emb_2)
    dist_euc = distancia_euclidiana(emb_1, emb_2)
    print(f"Comparando: '{p1}' e '{p2}'")
    print(f" -> Cosseno: {dist_cos:.4f} | Euclidiana: {dist_euc:.4f}")
print("-" * 40)

# ==========================================
# 4. Gerando o Gráfico 3D Interativo (PCA)
# ==========================================
print("📊 Renderizando o Gráfico 3D...")
nomes = list(dicionario_embeddings.keys())
vetores_originais = list(dicionario_embeddings.values())

pca = PCA(n_components=3)
vetores_3d = pca.fit_transform(vetores_originais)

df_plot = pd.DataFrame(vetores_3d, columns=['X', 'Y', 'Z'])
df_plot['Termo'] = nomes

fig = px.scatter_3d(
    df_plot, x='X', y='Y', z='Z', text='Termo',
    title="Visualização 3D dos Embeddings (Redução via PCA)",
    color='Termo'
)

fig.update_traces(marker=dict(size=8), textposition='top center')
fig.update_layout(showlegend=False)
fig.show()

--- TESTE COM TERMOS REAIS E GRÁFICO 3D ---
⏳ Gerando vetores na API (isso pode levar alguns segundos)...

Comparando: 'gato' e 'felino'
 -> Cosseno: 0.3637 | Euclidiana: 0.8528
Comparando: 'gato' e 'cachorro'
 -> Cosseno: 0.4789 | Euclidiana: 0.9784
Comparando: 'gato' e 'carro'
 -> Cosseno: 0.6193 | Euclidiana: 1.1130
Comparando: 'carro' e 'moto'
 -> Cosseno: 0.5660 | Euclidiana: 1.0642
Comparando: 'banana' e 'maça'
 -> Cosseno: 0.6016 | Euclidiana: 1.0968
Comparando: 'caminhão' e 'goiaba'
 -> Cosseno: 0.7120 | Euclidiana: 1.1934
----------------------------------------
📊 Renderizando o Gráfico 3D...


In [12]:
%%bash
# Entra na pasta do seu repositório
cd $NOME_REPO

# 1. Altera a mensagem do último commit (removendo a palavra "parcial")
git commit --amend -m "Entrega AULA_03: Geração de embeddings e gráfico 3D"

# 2. Força a atualização no GitHub para substituir a mensagem antiga
git push --force https://$GITHUB_TOKEN@github.com/$USUARIO_GITHUB/$NOME_REPO.git main

[main 6dfae2d] Entrega AULA_03: Geração de embeddings e gráfico 3D
 Date: Sat Aug 8 13:21:23 2026 +0000
 1 file changed, 425 insertions(+)
 create mode 100644 AULA_03/AULA_03_RAG.ipynb


To https://github.com/EridalgoRamos/Residencia-Trilhas-em-Tecnologias-IA-Generativa-RAG.git
 + 25b0388...6dfae2d main -> main (forced update)


In [13]:
import numpy as np
import pandas as pd
import requests
import os

print("⏳ Carregando modelos e gerando embeddings para as frases...\n")

# 1. Funções Base Otimizadas com NumPy
def get_embedding(texto):
    api_key = os.environ.get('OPENROUTER_API_KEY')
    response = requests.post(
        "https://openrouter.ai/api/v1/embeddings",
        headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
        json={"model": "openai/text-embedding-3-small", "input": texto}
    )
    return np.array(response.json()["data"][0]["embedding"], dtype=np.float32)

def similaridade_cosseno(emb_a, emb_b):
    produto_escalar = np.dot(emb_a, emb_b)
    norma_a, norma_b = np.linalg.norm(emb_a), np.linalg.norm(emb_b)
    if norma_a == 0 or norma_b == 0: return 0.0
    return produto_escalar / (norma_a * norma_b)

def distancia_cosseno(emb_a, emb_b):
    return 1.0 - similaridade_cosseno(emb_a, emb_b)

def distancia_euclidiana(emb_a, emb_b):
    return np.linalg.norm(emb_a - emb_b)

# 2. Definição dos Dados
frase_ancora = "O cachorro correu no parque e brincou com a bola."
frases_comparacao = [
    ("Similar (mesmo sentido)", "Um cão estava correndo no jardim e brincando com seu brinquedo."),
    ("Relacionado (animais)", "O gato dormiu na almofada da sala durante toda a tarde."),
    ("Diferente (economia)", "A taxa de juros do banco central subiu dois pontos percentuais."),
    ("Oposto/Negação", "Nenhum animal esteve no parque e o cão permaneceu preso em casa.")
]

# 3. Gerando os Vetores
vec_ancora = get_embedding(frase_ancora)
vecs_comp = [get_embedding(frase[1]) for frase in frases_comparacao]

# 4. Montando o DataFrame de Resultados
resultados = []
for (categoria, texto), vec in zip(frases_comparacao, vecs_comp):
    resultados.append({
        "Categoria": categoria,
        "Frase": texto,
        "Dist. Euclidiana": round(float(distancia_euclidiana(vec_ancora, vec)), 4),
        "Similaridade Cosseno": round(float(similaridade_cosseno(vec_ancora, vec)), 4),
        "Distância Cosseno": round(float(distancia_cosseno(vec_ancora, vec)), 4)
    })

df_resultados = pd.DataFrame(resultados)
# Ordenando da maior similaridade para a menor
df_resultados = df_resultados.sort_values(by="Similaridade Cosseno", ascending=False)
display(df_resultados)

⏳ Carregando modelos e gerando embeddings para as frases...



,Categoria,Frase,Dist. Euclidiana,Similaridade Cosseno,Distância Cosseno
0,Similar (mesmo sentido),Um cão estava correndo no jardim e brincando c...,0.6839,0.7661,0.2339
3,Oposto/Negação,Nenhum animal esteve no parque e o cão permane...,0.9265,0.5707,0.4293
1,Relacionado (animais),O gato dormiu na almofada da sala durante toda...,1.1139,0.3795,0.6205
2,Diferente (economia),A taxa de juros do banco central subiu dois po...,1.2395,0.2316,0.7684


In [14]:
import glob

# ====================================================
# CONFIGURAÇÃO DO CHUNKING (FATIAMENTO)
# Altere o SEPARADOR abaixo conforme a etapa da aula:
# Linha a linha = '\n'
# Parágrafos = '\n\n'
# Capítulos = '# '
# ====================================================
SEPARADOR_CHUNK = '\n\n'
NOME_REPO = os.environ.get('NOME_REPO', 'Residencia-Trilhas-em-Tecnologias-IA-Generativa-RAG')

print("🔍 Iniciando o Motor de Busca Semântica...\n")

# 1. Lendo os arquivos Markdown da Aula 02
caminho_arquivos = os.path.join(NOME_REPO, 'AULA_02', '*.md')
arquivos_md = glob.glob(caminho_arquivos)

if not arquivos_md:
    print("Atenção: Nenhum arquivo .md encontrado. Verifique se a pasta AULA_02 está correta.")
else:
    # Lendo o conteúdo de todos os documentos
    textos_completos = []
    for arq in arquivos_md:
        with open(arq, 'r', encoding='utf-8') as f:
            textos_completos.append(f.read())

    texto_unido = "\n".join(textos_completos)

    # 2. Fatiando o texto no separador escolhido
    chunks_brutos = texto_unido.split(SEPARADOR_CHUNK)
    # Limpando espaços em branco e ignorando blocos muito curtos (ex: linhas vazias)
    chunks_limpos = [c.strip() for c in chunks_brutos if len(c.strip()) > 50]

    print(f"📚 Documentos processados. Total de pedaços (chunks) gerados: {len(chunks_limpos)}")

    # 3. Definindo a Pergunta (Query)
    pergunta_usuario = "O que é opacidade algorítmica e como ela afeta as redes sociais?"
    print(f"❓ Sua Pergunta: '{pergunta_usuario}'\n")

    # 4. Gerando Embeddings (Para não sobrecarregar a API no teste manual, vamos usar apenas os primeiros 30 chunks)
    chunks_para_teste = chunks_limpos[:30]

    vec_pergunta = get_embedding(pergunta_usuario)

    resultados_busca = []
    for i, trecho in enumerate(chunks_para_teste):
        vec_trecho = get_embedding(trecho)
        score_similaridade = similaridade_cosseno(vec_pergunta, vec_trecho)
        resultados_busca.append((score_similaridade, trecho))

    # 5. Ordenando pelos maiores scores (Top 3)
    resultados_busca.sort(key=lambda x: x[0], reverse=True)
    top_3 = resultados_busca[:3]

    # 6. Exibindo os resultados
    print("🏆 TOP 3 TRECHOS ENCONTRADOS:\n")
    for ranking, (score, trecho) in enumerate(top_3, 1):
        print(f"--- [#{ranking}] Score de Similaridade: {score:.4f} ---")
        # Imprime apenas os 200 primeiros caracteres do chunk para não poluir a tela
        print(f"Texto: {trecho[:200]}...\n")

🔍 Iniciando o Motor de Busca Semântica...

📚 Documentos processados. Total de pedaços (chunks) gerados: 314
❓ Sua Pergunta: 'O que é opacidade algorítmica e como ela afeta as redes sociais?'

🏆 TOP 3 TRECHOS ENCONTRADOS:

--- [#1] Score de Similaridade: 0.5907 ---
Texto: Dessa maneira, a fim de facilitar a compreensão da dinâmica do diálogo entre algoritmos privados e a noção de espaço público , o artigo foi estruturado da seguinte forma: 1. O espaço público e as impr...

--- [#2] Score de Similaridade: 0.5589 ---
Texto: Este  artigo  investiga  a  dinâmica  das  redes  sociais  na  formação  do  debate público, com ênfase no Twitter (atualmente denominado X) e na influência dos  algoritmos  na  regulação  do  conteúd...

--- [#3] Score de Similaridade: 0.5534 ---
Texto: O objetivo da pesquisa, portanto, é identificar os impactos da construção dos algoritmos das redes como um espaço público e do engajamento de informações do Twitter/X  nas  decisões  e  acontecimentos...



In [15]:
%%bash
cd $NOME_REPO

# Assina o seu nome e e-mail
git config --global user.email "$EMAIL_GITHUB"
git config --global user.name "$USUARIO_GITHUB"

# Adiciona o notebook atualizado
git add AULA_03/

# Cria o pacote final
git commit -m "Entrega Final AULA_03: Motor de Busca Semântica e Análise com Pandas concluídos"

# Envia para a nuvem
git push https://$GITHUB_TOKEN@github.com/$USUARIO_GITHUB/$NOME_REPO.git main

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean


Everything up-to-date
